Objective: Prepare the crash dataset for machine-learning models while preventing data leakage and preserving information that would realistically be available for predicting accident severity.

In [1]:
import pandas as pd

df = pd.read_csv("../data/victorian_road_crash_data.csv")

df.shape

(200352, 52)

## 1. Inspect Target Distribution

We check the target classes to understand their distribution and identify class imbalance before modelling.

In [2]:
df["SEVERITY"].value_counts(dropna=False)

df["SEVERITY"].value_counts(normalize=True, dropna=False) * 100

SEVERITY
Other injury accident      62.361244
Serious injury accident    35.963704
Fatal accident              1.673055
Non injury accident         0.001996
Name: proportion, dtype: float64

### Remove Extremely Rare Class

The "Non injury accident" class contains only 4 records, so it cannot provide enough data for reliable model training.

In [3]:
df = df[df["SEVERITY"] != "Non injury accident"].copy()

### 2. Check for Target Leakage

We identify features that contain information about the outcome itself, which could make the model unrealistically accurate.

In [4]:
leakage_cols = [
    "INJ_OR_FATAL", "FATALITY", "SERIOUSINJURY",
    "OTHERINJURY", "NONINJURED"
]

df[leakage_cols].head()

,INJ_OR_FATAL,FATALITY,SERIOUSINJURY,OTHERINJURY,NONINJURED
0,1,0,0,1,2
1,1,0,0,1,2
2,1,0,0,1,4
3,1,0,1,0,1
4,1,0,1,0,0


### Remove Target Leakage

These columns directly describe crash injuries or fatalities, so they would reveal the target to the model.

In [5]:
leakage_cols = [
    "INJ_OR_FATAL", "FATALITY", "SERIOUSINJURY",
    "OTHERINJURY", "NONINJURED"
]

df = df.drop(columns=leakage_cols)

### 3. Separate Features and Target

We separate `SEVERITY` (the target) from the input features used for prediction.

In [6]:
X = df.drop(columns=["SEVERITY"])
y = df["SEVERITY"]

X.shape, y.shape

((200348, 46), (200348,))

### 4. Train-Test Split

We split the data before learning preprocessing parameters to prevent data leakage from the test set.

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape

((160278, 46), (40070, 46))

### 5. Identify Feature Types

We separate numerical and categorical features so each can receive appropriate preprocessing.

In [8]:
numeric_cols = X_train.select_dtypes(include="number").columns.tolist()
categorical_cols = X_train.select_dtypes(exclude="number").columns.tolist()

len(numeric_cols), len(categorical_cols)

(26, 20)

### 6. Check Missing Values

We check missing values in the training data before choosing an imputation strategy.

In [9]:
missing = X_train.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

SRNS                112226
RMA                   6152
DIVIDED               6152
ROAD_TYPE             2200
STAT_DIV_NAME          803
DEG_URBAN_NAME         786
ROAD_NAME              214
LGA_NAME                76
ROAD_ROUTE_1            71
LATITUDE                71
DTP_REGION              71
LONGITUDE               71
VICGRID_X               71
VICGRID_Y               71
NO_OF_VEHICLES           5
HEAVYVEHICLE             5
PT_VEHICLE               5
PASSENGERVEHICLE         5
MOTORCYCLE               5
dtype: int64

In [10]:
X_train[missing[missing > 0].index].dtypes

ROAD_NAME               str
ROAD_TYPE               str
ROAD_ROUTE_1        float64
LGA_NAME                str
DTP_REGION              str
LATITUDE            float64
LONGITUDE           float64
VICGRID_X           float64
VICGRID_Y           float64
NO_OF_VEHICLES      float64
HEAVYVEHICLE        float64
PASSENGERVEHICLE    float64
MOTORCYCLE          float64
PT_VEHICLE          float64
DEG_URBAN_NAME          str
SRNS                    str
RMA                     str
DIVIDED                 str
STAT_DIV_NAME           str
dtype: object

### 6. Extract Date & Time Features

We convert the accident date and time into useful numerical features for modelling.

In [11]:
X_train["ACCIDENT_TIME"] = pd.to_datetime(
    X_train["ACCIDENT_TIME"].astype(str),
    format="%H:%M:%S",
    errors="coerce"
)

X_test["ACCIDENT_TIME"] = pd.to_datetime(
    X_test["ACCIDENT_TIME"].astype(str),
    format="%H:%M:%S",
    errors="coerce"
)

X_train["HOUR"] = X_train["ACCIDENT_TIME"].dt.hour
X_test["HOUR"] = X_test["ACCIDENT_TIME"].dt.hour

In [12]:
X_train["HOUR"].value_counts().sort_index()

HOUR
0.0      2043
1.0      1720
2.0      1335
3.0      1229
4.0      1212
5.0      2172
6.0      5012
7.0      6894
8.0     10101
9.0      8297
10.0     8258
11.0     9243
12.0     9830
13.0     9546
14.0    10255
15.0    13222
16.0    13023
17.0    12963
18.0    10693
19.0     6741
20.0     4993
21.0     4630
22.0     3839
23.0     2956
Name: count, dtype: int64

### Remove Original Date & Time

The original date and time columns are no longer needed after extracting useful features from them.

In [13]:
X_train = X_train.drop(columns=["ACCIDENT_DATE", "ACCIDENT_TIME"])
X_test = X_test.drop(columns=["ACCIDENT_DATE", "ACCIDENT_TIME"])

In [14]:
X_train.shape, X_test.shape

((160278, 45), (40070, 45))

In [15]:
numeric_cols = X_train.select_dtypes(include="number").columns.tolist()
categorical_cols = X_train.select_dtypes(exclude="number").columns.tolist()

len(numeric_cols), len(categorical_cols)

(27, 18)

### 7. Handle Missing Numerical Values

Missing numerical values are replaced with the median calculated from the training data.

In [16]:
from sklearn.impute import SimpleImputer

num_imputer = SimpleImputer(strategy="median")

X_train[numeric_cols] = num_imputer.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = num_imputer.transform(X_test[numeric_cols])

In [17]:
X_train[numeric_cols].isnull().sum().sum()

np.int64(0)

### 8. Handle Missing Categorical Values

Missing categorical values are treated as a separate `"Missing"` category.

In [18]:
X_train[categorical_cols] = X_train[categorical_cols].fillna("Missing")
X_test[categorical_cols] = X_test[categorical_cols].fillna("Missing")

In [19]:
X_train[categorical_cols].isnull().sum().sum()

np.int64(0)

## 9. Categorical Feature Encoding

Categorical features must be converted into numerical representations before they can be used by machine learning models.

Low-cardinality categorical features are encoded using One-Hot Encoding. This creates a separate binary feature for each category.

`ROAD_NAME` is treated separately because it has very high cardinality (13,259 unique values). One-hot encoding it would create thousands of additional features and unnecessarily increase memory usage.

To preserve some information from `ROAD_NAME` without creating thousands of dummy variables, frequency encoding is used. Each road name is replaced by the number of times it appears in the training data.

All encoding transformations are learned from the training data and then applied to the test data to avoid data leakage.

In [20]:
X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()


# Remove identifier column
X_train_encoded = X_train_encoded.drop(columns=["ACCIDENT_NO"])
X_test_encoded = X_test_encoded.drop(columns=["ACCIDENT_NO"])

In [21]:
categorical_cols = X_train_encoded.select_dtypes(include=["str"]).columns.tolist()

print("Categorical features:", len(categorical_cols))
print(categorical_cols)

Categorical features: 17
['ACCIDENT_TYPE', 'DAY_OF_WEEK', 'DCA_CODE_DESCRIPTION', 'LIGHT_CONDITION', 'POLICE_ATTEND', 'ROAD_GEOMETRY', 'SPEED_ZONE', 'RUN_OFFROAD', 'ROAD_NAME', 'ROAD_TYPE', 'LGA_NAME', 'DTP_REGION', 'DEG_URBAN_NAME', 'SRNS', 'RMA', 'DIVIDED', 'STAT_DIV_NAME']


In [22]:
# Check the number of unique values in each categorical feature
categorical_cardinality = (
    X_train_encoded[categorical_cols]
    .nunique()
    .sort_values(ascending=False)
)

print(categorical_cardinality)

ROAD_NAME               13259
LGA_NAME                   88
DCA_CODE_DESCRIPTION       81
ROAD_TYPE                  79
SPEED_ZONE                 13
ACCIDENT_TYPE               9
DTP_REGION                  9
ROAD_GEOMETRY               9
DEG_URBAN_NAME              8
RMA                         7
DAY_OF_WEEK                 7
LIGHT_CONDITION             7
SRNS                        5
DIVIDED                     3
POLICE_ATTEND               3
STAT_DIV_NAME               3
RUN_OFFROAD                 2
dtype: int64


In [23]:
# Separate high-cardinality categorical features
high_cardinality_cols = ["ROAD_NAME"]

low_cardinality_cols = [
    col for col in categorical_cols
    if col not in high_cardinality_cols
]

print("Low-cardinality categorical features:", len(low_cardinality_cols))
print(low_cardinality_cols)

print("\nExcluded high-cardinality features:", high_cardinality_cols)

Low-cardinality categorical features: 16
['ACCIDENT_TYPE', 'DAY_OF_WEEK', 'DCA_CODE_DESCRIPTION', 'LIGHT_CONDITION', 'POLICE_ATTEND', 'ROAD_GEOMETRY', 'SPEED_ZONE', 'RUN_OFFROAD', 'ROAD_TYPE', 'LGA_NAME', 'DTP_REGION', 'DEG_URBAN_NAME', 'SRNS', 'RMA', 'DIVIDED', 'STAT_DIV_NAME']

Excluded high-cardinality features: ['ROAD_NAME']


In [24]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

X_train_cat = encoder.fit_transform(
    X_train_encoded[low_cardinality_cols]
)

X_test_cat = encoder.transform(
    X_test_encoded[low_cardinality_cols]
)

print("Encoded training shape:", X_train_cat.shape)
print("Encoded testing shape:", X_test_cat.shape)

Encoded training shape: (160278, 333)
Encoded testing shape: (40070, 333)


In [25]:
# Frequency encode ROAD_NAME using training data only

road_name_frequency = X_train_encoded["ROAD_NAME"].value_counts()

X_train_encoded["ROAD_NAME_FREQ"] = (
    X_train_encoded["ROAD_NAME"]
    .map(road_name_frequency)
)

X_test_encoded["ROAD_NAME_FREQ"] = (
    X_test_encoded["ROAD_NAME"]
    .map(road_name_frequency)
    .fillna(0)
)

print("ROAD_NAME frequency encoding completed.")
print("Train missing values:", X_train_encoded["ROAD_NAME_FREQ"].isna().sum())
print("Test missing values:", X_test_encoded["ROAD_NAME_FREQ"].isna().sum())

ROAD_NAME frequency encoding completed.
Train missing values: 0
Test missing values: 0


In [26]:
# Remove the original high-cardinality ROAD_NAME column
X_train_encoded = X_train_encoded.drop(columns=["ROAD_NAME"])
X_test_encoded = X_test_encoded.drop(columns=["ROAD_NAME"])

print("ROAD_NAME removed.")
print("Training shape:", X_train_encoded.shape)
print("Testing shape:", X_test_encoded.shape)

ROAD_NAME removed.
Training shape: (160278, 44)
Testing shape: (40070, 44)


In [27]:
print("Original numerical + encoded ROAD_NAME features:")
print(X_train_encoded.shape)

print("\nOne-hot encoded categorical features:")
print(X_train_cat.shape)

print("\nTest numerical + encoded ROAD_NAME features:")
print(X_test_encoded.shape)

print("\nTest one-hot encoded categorical features:")
print(X_test_cat.shape)

Original numerical + encoded ROAD_NAME features:
(160278, 44)

One-hot encoded categorical features:
(160278, 333)

Test numerical + encoded ROAD_NAME features:
(40070, 44)

Test one-hot encoded categorical features:
(40070, 333)


## 10. Numerical Feature Scaling

Numerical features can have very different ranges and magnitudes. Standardization transforms numerical features so that they have a mean of approximately 0 and a standard deviation of approximately 1.

The scaler is fitted only on the training data and then applied to the test data. This prevents information from the test set from influencing the preprocessing process.

In [28]:
numeric_cols_encoded = X_train_encoded.select_dtypes(
    include=["number"]
).columns.tolist()

print("Numerical features:", len(numeric_cols_encoded))
print(numeric_cols_encoded)

Numerical features: 28
['DCA_CODE', 'ROAD_ROUTE_1', 'LATITUDE', 'LONGITUDE', 'VICGRID_X', 'VICGRID_Y', 'TOTAL_PERSONS', 'MALES', 'FEMALES', 'BICYCLIST', 'PASSENGER', 'DRIVER', 'PEDESTRIAN', 'PILLION', 'MOTORCYCLIST', 'UNKNOWN', 'PED_CYCLIST_5_12', 'PED_CYCLIST_13_18', 'OLD_PED_65_AND_OVER', 'OLD_DRIVER_75_AND_OVER', 'YOUNG_DRIVER_18_25', 'NO_OF_VEHICLES', 'HEAVYVEHICLE', 'PASSENGERVEHICLE', 'MOTORCYCLE', 'PT_VEHICLE', 'HOUR', 'ROAD_NAME_FREQ']


In [29]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_num = scaler.fit_transform(
    X_train_encoded[numeric_cols_encoded]
)

X_test_num = scaler.transform(
    X_test_encoded[numeric_cols_encoded]
)

print("Scaled training numerical shape:", X_train_num.shape)
print("Scaled testing numerical shape:", X_test_num.shape)

Scaled training numerical shape: (160278, 28)
Scaled testing numerical shape: (40070, 28)


In [30]:
from scipy.sparse import hstack

X_train_final = hstack([
    X_train_num,
    X_train_cat
])

X_test_final = hstack([
    X_test_num,
    X_test_cat
])

print("Final training shape:", X_train_final.shape)
print("Final testing shape:", X_test_final.shape)

Final training shape: (160278, 361)
Final testing shape: (40070, 361)


## 11. Combining Numerical and Categorical Features

The scaled numerical features and one-hot encoded categorical features are combined into a single feature matrix.

A sparse matrix representation is maintained because the one-hot encoded data contains many zero values. This reduces unnecessary memory usage.

The resulting matrices contain 361 features:
- 28 scaled numerical features
- 333 one-hot encoded categorical features

In [31]:
from scipy.sparse import hstack

X_train_final = hstack([
    X_train_num,
    X_train_cat
])

X_test_final = hstack([
    X_test_num,
    X_test_cat
])

print("Final training shape:", X_train_final.shape)
print("Final testing shape:", X_test_final.shape)

Final training shape: (160278, 361)
Final testing shape: (40070, 361)


In [32]:
print("Training features:", X_train_final.shape[1])
print("Testing features:", X_test_final.shape[1])

Training features: 361
Testing features: 361


## 12. Target Encoding

The target variable `SEVERITY` contains categorical accident severity labels.

These labels are converted into numerical class labels using `LabelEncoder` so that they can be used by machine learning models.

The encoding is learned from the training target and then applied to the test target using the same mapping.

In [33]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

# Fit the encoder on training data and transform both sets
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

# Display the class mapping
print("Class mapping:")
for class_name, class_number in zip(
    label_encoder.classes_,
    range(len(label_encoder.classes_))
):
    print(f"{class_number} → {class_name}")

Class mapping:
0 → Fatal accident
1 → Other injury accident
2 → Serious injury accident


In [34]:
print("Encoded training target shape:", y_train_encoded.shape)
print("Encoded testing target shape:", y_test_encoded.shape)

print("\nTraining class distribution:")
print(pd.Series(y_train_encoded).value_counts().sort_index())

print("\nTesting class distribution:")
print(pd.Series(y_test_encoded).value_counts().sort_index())

Encoded training target shape: (160278,)
Encoded testing target shape: (40070,)

Training class distribution:
0     2682
1    99953
2    57643
Name: count, dtype: int64

Testing class distribution:
0      670
1    24989
2    14411
Name: count, dtype: int64


### Target Distribution After Encoding

The encoded training and testing targets preserve the class distribution from the original dataset.

The `Fatal accident` class remains substantially smaller than the `Other injury accident` and `Serious injury accident` classes. This class imbalance will be addressed later during the modelling stage.

The preprocessing stage does not apply any class-balancing technique so that the baseline model can be evaluated on the original class distribution.

## 13. Final Preprocessing Verification

The final preprocessing output is checked to ensure that the training and testing feature matrices are correctly shaped, contain no missing values, and remain aligned with their corresponding target variables.

The same preprocessing transformations have been applied consistently to the training and testing data, with all learned transformations fitted using training data only.

In [35]:
print("Training features:", X_train_final.shape)
print("Testing features:", X_test_final.shape)

print("Training target:", y_train_encoded.shape)
print("Testing target:", y_test_encoded.shape)

Training features: (160278, 361)
Testing features: (40070, 361)
Training target: (160278,)
Testing target: (40070,)


In [36]:
import numpy as np

print("Training NaN values:", np.isnan(X_train_final.data).sum())
print("Testing NaN values:", np.isnan(X_test_final.data).sum())

print("Training infinite values:", np.isinf(X_train_final.data).sum())
print("Testing infinite values:", np.isinf(X_test_final.data).sum())

Training NaN values: 0
Testing NaN values: 0
Training infinite values: 0
Testing infinite values: 0


In [37]:
assert X_train_final.shape[0] == len(y_train_encoded)
assert X_test_final.shape[0] == len(y_test_encoded)
assert X_train_final.shape[1] == X_test_final.shape[1]

print("✓ Training features and target are aligned")
print("✓ Testing features and target are aligned")
print("✓ Training and testing feature dimensions match")
print("✓ Final preprocessing verification passed")

✓ Training features and target are aligned
✓ Testing features and target are aligned
✓ Training and testing feature dimensions match
✓ Final preprocessing verification passed


## 14. Save Processed Data and Preprocessing Objects

The final preprocessed training and testing datasets are saved for use in the modelling stage.

The fitted preprocessing objects are also saved so that the same transformations can be reproduced consistently in future notebooks.

Only transformations fitted on the training data are used to process the test data.

In [38]:
from pathlib import Path

processed_dir = Path("../data")
processed_dir.mkdir(parents=True, exist_ok=True)

print("Saving processed data to:", processed_dir.resolve())

Saving processed data to: C:\Users\Rehan's Lenovo\OneDrive\Desktop\KLH\2-1\ML\github\Road-Accident-Severity-Prediction\data


In [39]:
from scipy.sparse import save_npz

save_npz(processed_dir / "X_train_final.npz", X_train_final)
save_npz(processed_dir / "X_test_final.npz", X_test_final)

print("Feature matrices saved.")

Feature matrices saved.


In [40]:
import numpy as np

np.save(processed_dir / "y_train_encoded.npy", y_train_encoded)
np.save(processed_dir / "y_test_encoded.npy", y_test_encoded)

print("Target arrays saved.")

Target arrays saved.


In [41]:
import numpy as np

np.save(processed_dir / "y_train_encoded.npy", y_train_encoded)
np.save(processed_dir / "y_test_encoded.npy", y_test_encoded)

print("Target arrays saved.")

Target arrays saved.


In [42]:
# Save final feature names

encoded_feature_names = encoder.get_feature_names_out(low_cardinality_cols)

final_feature_names = (
    numeric_cols_encoded
    + list(encoded_feature_names)
)

print("Number of feature names:", len(final_feature_names))
print("Number of final features:", X_train_final.shape[1])

Number of feature names: 361
Number of final features: 361


In [43]:
feature_names_df = pd.DataFrame({
    "feature_name": final_feature_names
})

feature_names_df.to_csv(
    processed_dir / "final_feature_names.csv",
    index=False
)

print("Feature names saved.")

Feature names saved.
